In [3]:
import pandas as pd

In [4]:
# Read data
orders = pd.read_excel(r"/content/global_mart.xlsx", sheet_name="orders")
transactions = pd.read_excel(r"/content/global_mart.xlsx", sheet_name="transactions")
new_orders = pd.read_excel(r"/content/new_orders.xlsx")


In [5]:
# Merge orders and transactions
df = orders.merge(transactions, on="order_id")

In [6]:
# Convert date
df["order_purchase_date"] = pd.to_datetime(df["order_purchase_date"])

In [7]:
# Keep only last 6 months of data
latest_date = df["order_purchase_date"].max()
last_6_months = latest_date - pd.DateOffset(months=6)

df = df[df["order_purchase_date"] >= last_6_months]

In [8]:
# Customer summary
summary = (
    df.groupby("customer_id")
      .agg(
          total_spend=("sales_amt", "sum"),
          number_of_orders=("order_id", "count")
      )
      .reset_index()
)

In [9]:
# Lambda to determine tier and discount
summary["Loyalty Tier"], summary["Discount Applicable"] = zip(
    *summary.apply(
        lambda x:
        ("Silver", 2) if x.total_spend < 500 and x.number_of_orders < 10 else
        ("Silver", 4) if x.total_spend < 500 else
        ("Gold", 6) if x.total_spend <= 2000 and x.number_of_orders < 10 else
        ("Gold", 8) if x.total_spend <= 2000 else
        ("Platinum", 10) if x.number_of_orders < 10 else
        ("Platinum", 15),
        axis=1
    )
)

print(summary)

    customer_id  total_spend  number_of_orders Loyalty Tier  \
0      AA-10315     2637.518                 2     Platinum   
1      AA-10375       29.320                 3       Silver   
2      AA-10480     2235.244                 6     Platinum   
3      AA-10645      230.895                 7       Silver   
4      AB-10015       77.144                 5       Silver   
..          ...          ...               ...          ...   
726    XP-21865     3004.570                20     Platinum   
727    YC-21895     1283.856                 3         Gold   
728    YS-21880     1475.120                 6         Gold   
729    ZC-21910      964.876                10         Gold   
730    ZD-21925     1114.545                 9         Gold   

     Discount Applicable  
0                     10  
1                      2  
2                     10  
3                      2  
4                      2  
..                   ...  
726                   15  
727                    6  


In [10]:
#Customer without deatils
customers_without_details = orders[orders['customer_id'].isnull()]
num_customers_without_details = customers_without_details['order_id'].nunique()

print(f"Number of orders from customers without details: {num_customers_without_details}")

Number of orders from customers without details: 8


In [11]:
# We use nunique() to count unique orders, as a single order might have multiple rows if it contains multiple items.
num_orders_last_six_months = df['order_id'].nunique()
print(f"Number of orders placed in the last six months: {num_orders_last_six_months}")


Number of orders placed in the last six months: 2020


In [16]:
#For total orders
df_filtered = df.dropna(subset=['customer_id', 'order_id','order_status','product_id'])
total_order_entries_last_six_months = len(df_filtered)
print(f"Total number of order entries (line items) in the last six months (excluding null customer_id and order_id): {total_order_entries_last_six_months}")

Total number of order entries (line items) in the last six months (excluding null customer_id and order_id): 4141


In [13]:
df_filtered.columns

Index(['order_id', 'customer_id', 'ship_mode', 'vendor_id', 'order_status',
       'order_purchase_date', 'order_approved_at',
       'order_delivered_carrier_date', 'order_delivered_customer_date',
       'order_estimated_delivery_date', 'id', 'product_id', 'sales_amt', 'qty',
       'discount', 'profit_amt'],
      dtype='object')

In [14]:
# Merge new_orders with the customer summary to get loyalty tier and discount information
new_orders_with_loyalty = new_orders.merge(summary[['customer_id', 'Loyalty Tier', 'Discount Applicable']], on='customer_id', how='left')

# Calculate the discounted price for new orders
new_orders_with_loyalty['Discounted Price'] = new_orders_with_loyalty['Original Price'] * (1 - new_orders_with_loyalty['Discount Applicable'] / 100)

print("New Orders with Loyalty Tiers and Discounted Prices:")
display(new_orders_with_loyalty)

New Orders with Loyalty Tiers and Discounted Prices:


,customer_id,Order_id,Original Price,Loyalty Tier,Discount Applicable,Discounted Price
0,AB-10105,CA-2014-103392,55.462,Gold,8.0,51.02504
1,DM-13525,CA-2014-103254,66.374,Gold,6.0,62.39156
2,RT-13456,CA-2014-103445,588.678,NaN,NaN,NaN
3,JH-10250,CA-2014-103136,224.731,NaN,NaN,NaN
4,AB-10105,CA-2014-103283,578.572,Gold,8.0,532.28624
5,DM-13525,CA-2014-103451,265.371,Gold,6.0,249.44874
6,JH-10250,CA-2014-103269,13.142,NaN,NaN,NaN
7,DM-13525,CA-2014-103451,524.170,Gold,6.0,492.71980
8,RT-13456,CA-2014-103326,133.676,NaN,NaN,NaN
9,DM-13525,CA-2014-103368,240.350,Gold,6.0,225.92900
